## Etape 1

Cellule des imports/constantes:

In [18]:
import pandas as pd

from src import config
from src.utils import find_input
from src.storage.export_features import EXPORT_PREFIX, SPLIT, TARGET, TRAIN, TEST

#toutes les colonnes visibles, 4 décimales
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
pd.set_option("display.float_format", "{:.4f}".format)   # 4 décimales, sans notation scientifique

Chargement de l'export feature et controle par rapport au schéma

In [11]:
import json

features_path = find_input(config.EXPORTS_DIR, EXPORT_PREFIX, ".parquet")
if features_path is None:
    raise FileNotFoundError(f"Aucun export {EXPORT_PREFIX}_<AAAAMMJJ>.parquet dans {config.EXPORTS_DIR}")

schema_path = features_path.with_suffix(".schema.json")
schema = json.loads(schema_path.read_text(encoding="utf-8"))

df = pd.read_parquet(features_path)
print(f"Fichier : {features_path.name}")
print(f"Lignes : {len(df):,} (schéma : {schema['rows']:,})".replace(",", " "))

# Effectifs par grade et par split, à comparer au schéma
counts = pd.crosstab(df[TARGET], df[SPLIT])
expected = pd.DataFrame(schema["split"]["counts"]).T[[TEST, TRAIN]].astype(int)
counts_match = counts[[TEST, TRAIN]].equals(expected)
print(f"Effectifs identiques au schéma : {counts_match}")

counts

Fichier : features_nutriscore_20261112.parquet
Lignes : 456 380 (schéma : 456 380)
Effectifs identiques au schéma : True


split,test,train
nutriscore_grade,,
a,12009,48035
b,9459,37835
c,19080,76320
d,24422,97688
e,26306,105226


Features, cible, et séparation train / test selon la colonne split figée

In [12]:
CATEGORICAL_FEATURES = ["aisle", "aisle2"]
NUMERIC_FEATURES = [
    *config.CORE_NUTRIENTS,               #7 nutriments obligatoires
    "fiber_100g",
    "fruits_vegetables_legumes_100g",
    "nova_group",                         #traité en nombre
    "additives_n",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

#colonnes à exclure: identifiant(code), split, target(nutriscore_grade) et nutriscore_score, garde fou pour plus tard
FORBIDDEN = {"code", SPLIT, TARGET, "nutriscore_score"}
#check features ne contiennent aucune de ces 4 colonnes sinon arrêt avec la liste des fautives
assert not FORBIDDEN & set(FEATURES), f"Colonne interdite dans les features : {FORBIDDEN & set(FEATURES)}"
print(f"{len(FEATURES)} features : {len(CATEGORICAL_FEATURES)} catégorielles, {len(NUMERIC_FEATURES)} numériques")

# Train seulement pour tout le travail
train = df[df[SPLIT] == TRAIN]
X_train, y_train = train[FEATURES], train[TARGET]
print(f"Train : {len(X_train):,} lignes".replace(",", " "))

# Types et part de valeurs manquantes de chaque feature, sur le train
pd.DataFrame({
    "dtype": X_train.dtypes.astype(str),
    "missing_pct": (X_train.isna().mean() * 100).round(2),
})

13 features : 2 catégorielles, 11 numériques
Train : 365 104 lignes


,dtype,missing_pct
aisle,str,0.00
aisle2,str,0.00
energy_kcal_100g,float64,0.00
fat_100g,float64,0.00
saturated_fat_100g,float64,0.00
carbohydrates_100g,float64,0.00
sugars_100g,float64,0.00
proteins_100g,float64,0.00
salt_100g,float64,0.00
fiber_100g,float64,46.87


Déséquilibre des classes dans le train, on affiche la part des différentes grade car les 5 pli de séparation seront fais sur celle ci, ca nous permet de mesurer le désiquilibre

In [13]:
class_counts = y_train.value_counts().sort_index()
class_share = (class_counts / len(y_train)).round(4)

imbalance = pd.DataFrame({"effectif": class_counts, "part": class_share})
print(f"Classe majoritaire : {class_counts.idxmax()} ({class_share.max():.2%})")
print(f"Classe minoritaire : {class_counts.idxmin()} ({class_share.min():.2%})")
print(f"Rapport majoritaire / minoritaire : {class_counts.max() / class_counts.min():.2f}")

imbalance

Classe majoritaire : e (28.82%)
Classe minoritaire : b (10.36%)
Rapport majoritaire / minoritaire : 2.78


,effectif,part
nutriscore_grade,,
a,48035,0.1316
b,37835,0.1036
c,76320,0.2090
d,97688,0.2676
e,105226,0.2882


Construction du Préprocesseur commun : numériques (imputation + indicateur, standardisation), catégorielles (one-hot)

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#seuil de fréquence du onehot: un rayon présent dans au moins 1 % des produits du train a sa propre colonne
#supposition de départ (0,01), à ajuster pour obtenir une soixantaine de colonnes (point de contrôle)
MIN_FREQUENCY = 0.01

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),   # médiane + colonne 0/1 « était manquante »
    ("scaler", StandardScaler()),                                       # moyenne 0, écart-type 1
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",          # un rayon inconnu au fit(étape d'entrainement) ne fait pas planter la prédiction
    min_frequency=MIN_FREQUENCY,      # regroupe les rayons rares
    sparse_output=False,              # tableau dense : plus simple à inspecter
)

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

# Contrôle du nombre de colonnes en sortie (ajusté sur le train seulement, jamais sur le test)
n_output_columns = preprocessor.fit(X_train).get_feature_names_out().size
print(f"{len(FEATURES)} colonnes brutes -> {n_output_columns} colonnes en sortie du préprocesseur")

13 colonnes brutes -> 55 colonnes en sortie du préprocesseur


Vérification des valeur de rayon

In [15]:
# Nombre de valeurs distinctes et part des rayons les plus rares, sur le train
for column in CATEGORICAL_FEATURES:
    shares = X_train[column].value_counts(normalize=True)
    print(f"{column} : {shares.size} valeurs distinctes ; les 5 plus rares :")
    print(shares.tail(5).round(5), "\n")

aisle : 11 valeurs distinctes ; les 5 plus rares :
aisle
en-fats-and-sauces          0.0672
en-beverages                0.0641
en-fruits-and-vegetables    0.0633
en-salty-snacks             0.0586
en-alcoholic-beverages      0.0001
Name: proportion, dtype: float64 

aisle2 : 39 valeurs distinctes ; les 5 plus rares :
aisle2
en-eggs                                0.0042
en-offals                              0.0034
en-artificially-sweetened-beverages    0.0033
en-fruit-nectars                       0.0014
en-waters-and-flavored-waters          0.0003
Name: proportion, dtype: float64 



Plis de la validation croisée, construits une seule fois et partagés par tous les modèles

In [16]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Contrôle : chaque pli de validation garde les proportions de grades du train
fold_shares = pd.DataFrame({
    f"pli {i}": y_train.iloc[val_idx].value_counts(normalize=True).sort_index()
    for i, (_, val_idx) in enumerate(cv.split(X_train, y_train), start=1)
}).round(4)
fold_shares

,pli 1,pli 2,pli 3,pli 4,pli 5
nutriscore_grade,,,,,
a,0.1316,0.1316,0.1316,0.1316,0.1316
b,0.1036,0.1036,0.1036,0.1036,0.1036
c,0.2090,0.2090,0.2090,0.2090,0.2090
d,0.2676,0.2676,0.2676,0.2676,0.2676
e,0.2882,0.2882,0.2882,0.2882,0.2882


Les 4 baselines en validation croisée 5 plis stratifiée

In [19]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
SCORING = ["f1_macro", "accuracy", "balanced_accuracy"]

baselines = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "tree_depth_3": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "tree_depth_8": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
}

cv_results = {}
for name, model in baselines.items():
    #un pipeline par modèle : préprocesseur neuf (clone) puis modèle, réajustés à chaque pli
    pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("model", model)])
    cv_results[name] = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=SCORING)
    print(f"{name} : terminé")

# Tableau comparatif : moyenne et écart-type sur les 5 plis
baseline_table = pd.DataFrame([
    {
        "modele": name,
        **{f"{metric}_mean": res[f"test_{metric}"].mean() for metric in SCORING},
        **{f"{metric}_std": res[f"test_{metric}"].std() for metric in SCORING},
        "fit_time_s": res["fit_time"].mean(),
        "score_time_s": res["score_time"].mean(),
    }
    for name, res in cv_results.items()
]).set_index("modele")
baseline_table

dummy_most_frequent : terminé
dummy_stratified : terminé
tree_depth_3 : terminé
tree_depth_8 : terminé


,f1_macro_mean,accuracy_mean,balanced_accuracy_mean,f1_macro_std,accuracy_std,balanced_accuracy_std,fit_time_s,score_time_s
modele,,,,,,,,
dummy_most_frequent,0.0895,0.2882,0.2000,0.0000,0.0000,0.0000,0.9157,0.2788
dummy_stratified,0.1992,0.2257,0.1992,0.0017,0.0016,0.0017,1.4218,0.6425
tree_depth_3,0.3600,0.5234,0.4138,0.0003,0.0004,0.0004,3.8243,0.7706
tree_depth_8,0.7444,0.7784,0.7419,0.0013,0.0012,0.0010,5.3456,0.7159


l’accuracy de 0,29 est plus de trois fois supérieure au F1 macro de 0,09. L’accuracy donne une impression de score correct, alors que le modèle ignore 4 grades sur 5.

on a désormais le tableau comparatif des différentes baselines, on va pouvoir l'utiliser pour le comparer a notre modèle qui sera entrainé